# NLP Preprocessing Engine
### Data Science Internship – February 2026
**Task: Build a Robust NLP Preprocessing Engine (Advanced)**

---
## Install & Import Libraries

In [1]:
# Install required library
!pip install nltk

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.5 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.5 kB ? eta -:--:--
     ------------------ ------------------- 20.5/41.5 kB 131.3 kB/s eta 0:00:01
     ---------------------------- --------- 30.7/41.5 kB 146.3 kB/s eta 0:00:01
     -------------------------------------  41.0/41.5 kB 164.3 kB/s eta 0:00:01
     -------------------------------------- 41.5/41.5 kB 133.5 kB/s eta 0:00:00
     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     -------------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import string
import nltk
from collections import Counter

# Download required NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("All libraries imported successfully!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Zaara\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Zaara\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Zaara\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Zaara\AppData\Roaming\nltk_data...


All libraries imported successfully!


---
## Task 1: Conceptual Understanding (Written Answers)

### Q1. What is the difference between "Love" and "love" in NLP?

In NLP, **"Love"** and **"love"** are treated as two different tokens by default because computers are case-sensitive. When we write "Love" with a capital L (usually at the start of a sentence) and "love" in lowercase, the model sees them as completely different words — even though they carry the same meaning. This is why **lowercasing** is a critical preprocessing step: it normalizes both to "love", reducing vocabulary size and helping the model understand that they represent the same concept.

---

### Q2. What happens if stopwords are not removed?

If stopwords (like "is", "the", "am", "are", "in") are not removed:
- They **dominate** the token frequency, drowning out the actually meaningful words.
- ML models may **overfit** on these common but meaningless words.
- The **vocabulary size** becomes unnecessarily large, increasing computation.
- Features extracted for tasks like **sentiment analysis or topic modeling** become noisy and less accurate.

---

### Q3. Two real-world scenarios where removing stopwords can be HARMFUL:

**Scenario 1 – Sentiment Analysis with Negation:**  
Consider "I am **not** happy" vs "I am happy". If we remove the stopword **"not"**, both sentences become "happy", completely reversing the sentiment. This leads to totally wrong predictions in sentiment analysis.

**Scenario 2 – Question Answering / Information Retrieval:**  
In a search query like **"Who is the President of India?"**, words like "who", "is", "the", "of" are technically stopwords — but removing them leaves behind "President India", which changes the intent of the query and may return wrong results.

---

### Q4. Difference between Stemming and Lemmatization:

| Feature | Stemming | Lemmatization |
|---|---|---|
| Approach | Chops off word endings (crude rules) | Uses vocabulary + grammar context |
| Output | May not be a real word | Always a valid dictionary word |
| Speed | Faster | Slower (but more accurate) |
| Example | "running" → "run**n**" | "running" → "run" |
| Example | "studies" → "studi" | "studies" → "study" |
| Best for | Search engines (speed matters) | NLP tasks needing accuracy |

---
## Task 2: Build an Advanced Preprocessing Function

In [3]:
# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Words to KEEP even if they are short or stopwords (meaningful negation words)
KEEP_WORDS = {"no", "not"}


def preprocess_text(text):
    """
    Advanced NLP Preprocessing Function.
    
    Steps:
    1. Handle empty or non-string input
    2. Remove URLs and email-like patterns
    3. Remove emojis and non-ASCII characters
    4. Convert to lowercase
    5. Handle repeated characters (e.g. 'loooove' → 'love')
    6. Remove punctuation and special characters
    7. Remove numbers
    8. Tokenize
    9. Remove extra spaces / empty tokens
    10. Remove stopwords (but keep meaningful words like 'no', 'not')
    11. Remove very short tokens (length <= 2), except KEEP_WORDS
    12. Lemmatize tokens
    
    Returns: list of cleaned tokens
    """
    
    # ── Step 1: Handle edge cases ──────────────────────────────────────────
    if not isinstance(text, str) or text.strip() == "":
        return []  # Return empty list for empty/invalid input
    
    # ── Step 2: Remove URLs (http, https, www) and email patterns ──────────
    text = re.sub(r'http\S+|https\S+|www\.\S+', '', text)   # URLs
    text = re.sub(r'\S+@\S+\.\S+', '', text)                 # Emails
    
    # ── Step 3: Remove emojis and non-ASCII characters ─────────────────────
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # ── Step 4: Convert to lowercase ───────────────────────────────────────
    text = text.lower()
    
    # ── Step 5: Handle repeated characters (3+ same chars → 1) ─────────────
    # e.g. 'soooo' → 'so', 'goooood' → 'good', 'loooove' → 'love'
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # ── Step 6: Remove punctuation and special characters ──────────────────
    text = re.sub(r'[^\w\s]', '', text)  # Keep only word chars and spaces
    
    # ── Step 7: Remove numbers ─────────────────────────────────────────────
    text = re.sub(r'\d+', '', text)
    
    # ── Step 8: Tokenize (split into words) ────────────────────────────────
    tokens = text.split()
    
    # ── Step 9: Remove stopwords (keep KEEP_WORDS like 'no', 'not') ────────
    tokens = [t for t in tokens if t not in stop_words or t in KEEP_WORDS]
    
    # ── Step 10: Remove very short tokens (len <= 2), except KEEP_WORDS ────
    tokens = [t for t in tokens if len(t) > 2 or t in KEEP_WORDS]
    
    # ── Step 11: Lemmatize each token ──────────────────────────────────────
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return tokens


# Quick test
print("Test 1:", preprocess_text("I have 2 dogs"))
print("Test 2:", preprocess_text("soooo goooood!!!"))
print("Test 3:", preprocess_text("Visit http://example.com now"))
print("Test 4:", preprocess_text("WOW!!! This is GREAT!!!"))
print("Test 5:", preprocess_text(""))        # Empty string
print("Test 6:", preprocess_text("😍😍😍"))   # Only emojis
print("Test 7:", preprocess_text("123456"))  # Only numbers

Test 1: ['dog']
Test 2: ['god']
Test 3: ['visit']
Test 4: ['wow', 'great']
Test 5: []
Test 6: []
Test 7: []


---
## Task 3: Stress Testing (10 Diverse Sentences)

In [4]:
# Sample inputs from the assignment
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

print("=" * 70)
print("TASK 3: STRESS TESTING RESULTS")
print("=" * 70)

all_results = []  # Store results for use in later tasks

for i, sentence in enumerate(sample_inputs, 1):
    tokens = preprocess_text(sentence)
    clean_sentence = ' '.join(tokens)
    
    all_results.append({
        'original': sentence,
        'tokens': tokens,
        'clean_sentence': clean_sentence
    })
    
    print(f"\n Sentence {i}")
    print(f"   Original Text    : {sentence}")
    print(f"   Cleaned Tokens   : {tokens}")
    print(f"   Cleaned Sentence : {clean_sentence}")
    print("-" * 70)

TASK 3: STRESS TESTING RESULTS

 Sentence 1
   Original Text    : Get 100% FREE access now!!!
   Cleaned Tokens   : ['get', 'free', 'access']
   Cleaned Sentence : get free access
----------------------------------------------------------------------

 Sentence 2
   Original Text    : I absolutely looooved this product 😍😍
   Cleaned Tokens   : ['absolutely', 'loved', 'product']
   Cleaned Sentence : absolutely loved product
----------------------------------------------------------------------

 Sentence 3
   Original Text    : Worst service ever... 0/10
   Cleaned Tokens   : ['worst', 'service', 'ever']
   Cleaned Sentence : worst service ever
----------------------------------------------------------------------

 Sentence 4
   Original Text    : Call me at 9876543210
   Cleaned Tokens   : ['call']
   Cleaned Sentence : call
----------------------------------------------------------------------

 Sentence 5
   Original Text    : This is THE best course!!!
   Cleaned Tokens   : ['best

---
## Task 4: Token Analytics

In [5]:
print("=" * 70)
print("TASK 4: TOKEN ANALYTICS")
print("=" * 70)

most_noise_sentence = None
most_noise_count = -1
most_meaningful_sentence = None
most_meaningful_count = -1

for i, result in enumerate(all_results, 1):
    tokens = result['tokens']
    original = result['original']
    
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_token_len = round(sum(len(t) for t in tokens) / total_tokens, 2) if total_tokens > 0 else 0
    
    # Calculate noise: original word count minus cleaned token count
    original_word_count = len(original.split())
    noise_removed = original_word_count - total_tokens
    
    print(f"\n Sentence {i}: \"{original}\"")
    print(f"   Total Tokens     : {total_tokens}")
    print(f"   Unique Tokens    : {unique_tokens}")
    print(f"   Avg Token Length : {avg_token_len} characters")
    print(f"   Noise Removed    : {noise_removed} words")
    
    # Track the most noisy
    if noise_removed > most_noise_count:
        most_noise_count = noise_removed
        most_noise_sentence = (i, original)
    
    # Track most meaningful (most tokens retained)
    if total_tokens > most_meaningful_count:
        most_meaningful_count = total_tokens
        most_meaningful_sentence = (i, original)

print("\n" + "=" * 70)
print(" ANALYSIS RESULTS")
print("=" * 70)
print(f"\n Most Noisy Sentence     : Sentence {most_noise_sentence[0]}")
print(f"   → \"{most_noise_sentence[1]}\"")
print(f"   ({most_noise_count} words removed as noise)")
print(f"\n Most Meaningful Sentence: Sentence {most_meaningful_sentence[0]}")
print(f"   → \"{most_meaningful_sentence[1]}\"")
print(f"   ({most_meaningful_count} meaningful tokens retained)")

TASK 4: TOKEN ANALYTICS

 Sentence 1: "Get 100% FREE access now!!!"
   Total Tokens     : 3
   Unique Tokens    : 3
   Avg Token Length : 4.33 characters
   Noise Removed    : 2 words

 Sentence 2: "I absolutely looooved this product 😍😍"
   Total Tokens     : 3
   Unique Tokens    : 3
   Avg Token Length : 7.33 characters
   Noise Removed    : 3 words

 Sentence 3: "Worst service ever... 0/10"
   Total Tokens     : 3
   Unique Tokens    : 3
   Avg Token Length : 5.33 characters
   Noise Removed    : 1 words

 Sentence 4: "Call me at 9876543210"
   Total Tokens     : 1
   Unique Tokens    : 1
   Avg Token Length : 4.0 characters
   Noise Removed    : 3 words

 Sentence 5: "This is THE best course!!!"
   Total Tokens     : 2
   Unique Tokens    : 2
   Avg Token Length : 5.0 characters
   Noise Removed    : 3 words

 Sentence 6: "Visit https://openai.com now!"
   Total Tokens     : 1
   Unique Tokens    : 1
   Avg Token Length : 5.0 characters
   Noise Removed    : 2 words

 Sentence 7: "

---
## Task 5: Frequency Analysis

In [6]:
from collections import Counter

# Combine ALL tokens from all sentences into one flat list
all_tokens = []
for result in all_results:
    all_tokens.extend(result['tokens'])

# Count frequency of each token
token_counts = Counter(all_tokens)

print("=" * 70)
print("TASK 5: FREQUENCY ANALYSIS")
print("=" * 70)

print(f"\nTotal tokens across all sentences : {len(all_tokens)}")
print(f"Unique tokens across all sentences: {len(token_counts)}")

# Top 10 most frequent words
print("\n Top 10 Most Frequent Words:")
print("-" * 35)
for rank, (word, count) in enumerate(token_counts.most_common(10), 1):
    print(f"   {rank:2}. '{word}' → {count} time(s)")

# Top 5 least frequent words
print("\n Top 5 Least Frequent Words:")
print("-" * 35)
least_common = token_counts.most_common()[:-6:-1]  # last 5 items reversed
for rank, (word, count) in enumerate(least_common, 1):
    print(f"   {rank}. '{word}' → {count} time(s)")

TASK 5: FREQUENCY ANALYSIS

Total tokens across all sentences : 21
Unique tokens across all sentences: 21

 Top 10 Most Frequent Words:
-----------------------------------
    1. 'get' → 1 time(s)
    2. 'free' → 1 time(s)
    3. 'access' → 1 time(s)
    4. 'absolutely' → 1 time(s)
    5. 'loved' → 1 time(s)
    6. 'product' → 1 time(s)
    7. 'worst' → 1 time(s)
    8. 'service' → 1 time(s)
    9. 'ever' → 1 time(s)
   10. 'call' → 1 time(s)

 Top 5 Least Frequent Words:
-----------------------------------
   1. 'happy' → 1 time(s)
   2. 'not' → 1 time(s)
   3. 'offer' → 1 time(s)
   4. 'limited' → 1 time(s)
   5. 'win' → 1 time(s)


---
## Task 6: Build Full Pipeline

In [7]:
def full_pipeline(text_list):
    """
    Full NLP Preprocessing Pipeline.
    
    Takes a list of raw text strings and returns a dictionary with:
    - 'tokens'         : list of token lists (one per input sentence)
    - 'clean_sentences': list of cleaned sentences (one per input)
    
    Args:
        text_list (list): List of raw text strings
    
    Returns:
        dict: {'tokens': [...], 'clean_sentences': [...]}
    """
    
    # Validate input is a list
    if not isinstance(text_list, list):
        raise TypeError("Input must be a list of strings.")
    
    all_tokens = []
    clean_sentences = []
    
    for text in text_list:
        tokens = preprocess_text(text)       # Run preprocessing on each text
        clean_sentence = ' '.join(tokens)    # Join tokens into a sentence
        
        all_tokens.append(tokens)
        clean_sentences.append(clean_sentence)
    
    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }


# ── Run the full pipeline on sample inputs ──────────────────────────────────
pipeline_output = full_pipeline(sample_inputs)

print("=" * 70)
print("TASK 6: FULL PIPELINE OUTPUT")
print("=" * 70)

print("\n Pipeline Result Dictionary:")
print("\n Key: 'tokens'")
for i, tokens in enumerate(pipeline_output['tokens'], 1):
    print(f"   [{i}] {tokens}")

print("\n Key: 'clean_sentences'")
for i, sentence in enumerate(pipeline_output['clean_sentences'], 1):
    print(f"   [{i}] '{sentence}'")

TASK 6: FULL PIPELINE OUTPUT

 Pipeline Result Dictionary:

 Key: 'tokens'
   [1] ['get', 'free', 'access']
   [2] ['absolutely', 'loved', 'product']
   [3] ['worst', 'service', 'ever']
   [4] ['call']
   [5] ['best', 'course']
   [6] ['visit']
   [7] ['no', 'bad']
   [8] ['got']
   [9] ['win', 'limited', 'offer']
   [10] ['not', 'happy']

 Key: 'clean_sentences'
   [1] 'get free access'
   [2] 'absolutely loved product'
   [3] 'worst service ever'
   [4] 'call'
   [5] 'best course'
   [6] 'visit'
   [7] 'no bad'
   [8] 'got'
   [9] 'win limited offer'
   [10] 'not happy'


---
## Task 7: Error Handling

In [8]:
print("=" * 70)
print("TASK 7: ERROR HANDLING – EDGE CASES")
print("=" * 70)

edge_cases = [
    ("",           "Empty String"),
    ("😍🔥💯😂",   "Only Emojis"),
    ("1234567890", "Only Numbers"),
    ("   ",        "Only Whitespace"),
    (None,         "None input"),
]

for text, label in edge_cases:
    try:
        result = preprocess_text(text)
        clean = ' '.join(result) if result else '[No meaningful tokens found]'
        print(f"\n Edge Case  : {label}")
        print(f"   Input       : {repr(text)}")
        print(f"   Tokens      : {result}")
        print(f"   Output      : {clean}")
    except Exception as e:
        print(f"\n Error for '{label}': {e}")

TASK 7: ERROR HANDLING – EDGE CASES

 Edge Case  : Empty String
   Input       : ''
   Tokens      : []
   Output      : [No meaningful tokens found]

 Edge Case  : Only Emojis
   Input       : '😍🔥💯😂'
   Tokens      : []
   Output      : [No meaningful tokens found]

 Edge Case  : Only Numbers
   Input       : '1234567890'
   Tokens      : []
   Output      : [No meaningful tokens found]

 Edge Case  : Only Whitespace
   Input       : '   '
   Tokens      : []
   Output      : [No meaningful tokens found]

 Edge Case  : None input
   Input       : None
   Tokens      : []
   Output      : [No meaningful tokens found]
